# 🧮 Option Pricing Engine

This notebook implements a versatile option pricing engine capable of valuing both **European** and **American** options using:

- **Black-Scholes Closed-Form Solution** for European options  
- **Binomial Tree Model** for flexible and intuitive pricing  


In [1]:
import numpy as np
from scipy.stats import norm
import copy

class Option:
    def __init__(self, 
                 S0 : float, 
                 K : float, 
                 r : float,
                 sigma : float,
                 T : float, 
                 is_call : bool,
                 is_american : bool):
        self.S0 = S0
        self.K = K
        self.r = r
        self.sigma = sigma
        self.T = T
        self.is_call = is_call
        self.is_american = is_american
    
    def payoff(self, S):
        mult = 1 if self.is_call else -1
        return max(mult * (S - self.K), 0.0)
    
    def discount(self, dt):
        return np.exp(-self.r * dt)

class OptionPricer:
    def price(self, inst : Option) -> float:
        raise NotImplementedError
    
class BlackSholesPricer(OptionPricer):
        
    def price(self, inst : Option) -> float:
        if inst.is_american:
            raise RuntimeError("BlackSholesPricer does not support American Option")

        d1 = (np.log(inst.S0 / inst.K) + (inst.r + (inst.sigma * inst.sigma) / 2) * inst.T) / (inst.sigma * np.sqrt(inst.T))
        d2 = d1 - inst.sigma * np.sqrt(inst.T)

        call = norm.cdf(d1) * inst.S0 - norm.cdf(d2) * inst.K * inst.discount(inst.T)
        if inst.is_call:
            return call
        else:
            #deal with put American Option
            return call - inst.S0 + inst.K * inst.discount(inst.T)

        
class BinomialTreePricer(OptionPricer):
    def __init__(self, N_per_year : int = 252) -> None:
        self.N_per_year = N_per_year

    def price(self, inst : Option) -> float:

        N = int(self.N_per_year * inst.T)
        dt = 1.0 / N
        u = np.exp(inst.r * dt + inst.sigma * np.sqrt(dt))
        d = np.exp(inst.r * dt - inst.sigma * np.sqrt(dt))
        p = (np.exp(inst.r * dt) - d) / (u - d)
        df = inst.discount(dt)

        SGrid = [np.zeros(i+1) for i in range(N+1)]
        VGrid = [-1 * np.ones(i+1) for i in range(N+1)] # -1 for not evaluated

        def value_recursive(i, j, S):
            SGrid[i][j] = S
            if VGrid[i][j] != -1:
                return VGrid[i][j] # already evaluated, return directly
            if i == N:
                # reach T, calculate the final payoff
                VGrid[i][j] = inst.payoff(S)
                return VGrid[i][j]
            
            holding_value = df * (p * value_recursive(i+1, j+1, S * u) + \
                                  (1.0 - p) * value_recursive(i+1, j, S * d))
            if not inst.is_american:
                VGrid[i][j] = holding_value
            else:
                exercise_value = inst.payoff(S)
                VGrid[i][j] = max(holding_value, exercise_value)
            return VGrid[i][j]
        
        value_recursive(0, 0, inst.S0)
        return VGrid[0][0]

In [5]:
tree_pricer = BinomialTreePricer()
bs_pricer = BlackSholesPricer()

# Define option parameters
S0           = 100      # Initial stock price
K            = 100      # Strike price
r            = 0.01     # Risk-free interest rate
sigma        = 0.30     # Volatility
T            = 1.0      # Time to maturity (in years)
is_call      = False    # False = put option
is_american  = True     # True = American-style option

# Create the option instance
american_option = Option(S0, K, r, sigma, T, is_call, is_american)
eu_option = Option(S0, K, r, sigma, T, is_call, not is_american)

# Print option parameters
print("Option Parameters")
print(f"{'Type:':20} {'Call' if is_call else 'Put'}")
print(f"{'Style:':20} {'American' if is_american else 'European'}")
print(f"{'Initial Price (S0):':20} {S0}")
print(f"{'Strike Price (K):':20} {K}")
print(f"{'Risk-Free Rate (r):':20} {r}")
print(f"{'Volatility (sigma):':20} {sigma}")
print(f"{'Time to Maturity (T):':20} {T} year(s)")

# Price the option
tree_price_am = tree_pricer.price(american_option)
tree_price_eu = tree_pricer.price(eu_option)
bs_price = bs_pricer.price(eu_option)

# Print pricing results
print("\nOption Pricing Results")
print(f"{'Binomial Tree (AM):':20} {tree_price_am:.4f}")
print(f"{'Binomial Tree (EU):':20} {tree_price_eu:.4f}")
print(f"{'Black-Scholes (EU):':20} {bs_price:.4f}")

Option Parameters
Type:                Put
Style:               American
Initial Price (S0):  100
Strike Price (K):    100
Risk-Free Rate (r):  0.01
Volatility (sigma):  0.3
Time to Maturity (T): 1.0 year(s)

Option Pricing Results
Binomial Tree (AM):  11.4544
Binomial Tree (EU):  11.3796
Black-Scholes (EU):  11.3733


In [7]:
print("-----end------")

-----end------
